In [41]:
import os
import random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
import json

In [42]:
SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

def tokenize(sentence):
    lst = []
    for word in sentence.lower().split():
        lst.append(word)
    return lst

def is_tag_line(line):
    line = line.strip()
    return line.startswith("<") and line.endswith(">") 

def is_stage_direction(line):
    line = line.strip()
    return line.startswith("(") and line.endswith(")")

def load_corpus(path, min_len=4, max_len=30, max_sentences=50000):
    lst = []
    with open(path, "r") as f:
        for line in f:
            if is_tag_line(line) or (not line.strip()) or is_stage_direction(line):
                continue
            tokens = tokenize(line)
            if min_len <= len(tokens) <= max_len:
                sentence = ' '.join(tokens)
                lst.append(sentence)
                    
            if len(lst) >= max_sentences:
                break
    return lst



def load_corpus_from_dir(dir_path, min_len=4, max_len=30, max_sentences=50000):
    lst = []
    # TODO: 目录读取
    for file in sorted(os.listdir(dir_path)):
        if not file.endswith(".txt"):
            continue
        file_path= os.path.join(dir_path, file)
        remaining = max_sentences - len(lst)
        sentences = load_corpus(path=file_path, min_len=min_len, max_len=max_len, max_sentences=remaining)
        lst.extend(sentences)
        if len(lst) >= max_sentences:
            break
    return lst
    


def build_vocab(sentences, min_freq=2):
    word2idx = {"<PAD>":0, "<SOS>":1, "<EOS>":2, "<UNK>":3}
    idx2word = {0:"<PAD>", 1:"<SOS>", 2:"<EOS>", 3:"<UNK>"}
    lst_freq= {}
    
    for sentence in sentences:
        for token in tokenize(sentence):
            if token not in lst_freq:
                lst_freq[token] = 1
            else:
                lst_freq[token] += 1
    lst_freq = {k:v for k,v in lst_freq.items() if v >= min_freq}
    sorted_freq = sorted(lst_freq.items(), key=lambda x:(-x[1],x[0]))
    for token,freq in sorted_freq:
        idx = len(word2idx)
        word2idx[token] = idx
        idx2word[idx] = token
    return word2idx, idx2word

def encode(sentence, word2idx):
    enc = [word2idx["<SOS>"]]
    for token in tokenize(sentence):
        if token in word2idx:
            enc.append(word2idx[token])
        else:
            enc.append(word2idx["<UNK>"])
    enc.append(word2idx["<EOS>"])
    return enc

def decode(ids, idx2word):
    sentence = ''
    for idx in ids:
        if idx not in (0,1,2):
            sentence += idx2word[idx] + ' '
    return sentence.strip()


In [43]:
class TextDataset(Dataset):
    def __init__(self, sentences, word2idx):
        self.sentences = sentences
        self.word2idx = word2idx


    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        ids = encode(self.sentences[idx], self.word2idx)
        return ids, len(ids)

def collate_fn(batch):
    max_len = 1
    padded_ids , lengths = [], []
    pad_id = 0
    for _,length in batch:
        if length>max_len:
            max_len = length
        lengths.append(length)
    for ids,_ in batch:
        padded_ids.append(ids + [pad_id]*(max_len-len(ids)))
    padded_ids = torch.LongTensor(padded_ids)
    lengths = torch.LongTensor(lengths)
    return padded_ids, lengths

def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


In [44]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, pad_idx=0):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx,
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

    def forward(self, src_ids, src_lengths):
        # src_ids: [batch_size, seq_len]
        # src_lengths: [batch_size]
        
        embedded = self.embedding(src_ids)
        packed_embedded = pack_padded_sequence(
            input=embedded,
            lengths=src_lengths,
            batch_first=True,
            enforce_sorted=False,
        )
        output, (hidden, cell) = self.lstm(packed_embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx,
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tgt_input_ids, hidden, cell):
        # tgt_input_ids: [batch_size, tgt_len]
        # hidden/cell: Encoder 输出，用作 Decoder 初始状态
        embeded = self.embedding(tgt_input_ids)
        outputs, (hidden, cell) = self.lstm(embeded, (hidden, cell))
        logits = self.fc(outputs)
        return logits,(hidden, cell)

In [45]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src_ids, src_lengths):
        hidden, cell = self.encoder(src_ids, src_lengths)
        decoder_input = src_ids[:, :-1]
        decoder_target = src_ids[:, 1:]
        logits,_ = self.decoder(decoder_input, hidden, cell)
        return logits, decoder_target


In [46]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch_ids, batch_lengths in loader:
        batch_ids = batch_ids.to(device)
        batch_lengths = batch_lengths.cpu()
        optimizer.zero_grad()
        logits, target = model(batch_ids, batch_lengths)
        vocab_size = logits.shape[-1]
        logits_flat = logits.reshape(-1, vocab_size)
        target_flat = target.reshape(-1)
        loss = criterion(logits_flat, target_flat)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch_ids, batch_lengths in loader:
            batch_ids = batch_ids.to(device)
            batch_lengths = batch_lengths.cpu()
            logits, target = model(batch_ids, batch_lengths)
            vocab_size = logits.shape[-1]
            logits_flat = logits.reshape(-1, vocab_size)
            target_flat = target.reshape(-1)
            loss = criterion(logits_flat, target_flat)
            total_loss += loss.item()

    return total_loss / len(loader)

def greedy_decode(model, sentence, word2idx, idx2word, max_len=32):
    model.eval()
    device = next(model.parameters()).device
    ids = encode(sentence, word2idx)
    src_lengths = torch.tensor([len(ids)], dtype=torch.long).cpu()
    ids = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        hidden,cell = model.encoder(ids, src_lengths)
        decoder_input = torch.tensor([[word2idx["<SOS>"]]], dtype=torch.long)
        generated_ids = []
        for _ in range(max_len):
            logits, (hidden, cell) = model.decoder(decoder_input, hidden, cell)
            next_id = logits[:,-1,:].argmax().item()
            if next_id == word2idx["<EOS>"]:
                break
            generated_ids.append(next_id)
            decoder_input = torch.tensor([[next_id]], dtype=torch.long)
    return decode(generated_ids, idx2word)


In [47]:
def split_sentences(sentences, train_ratio=0.8, val_ratio=0.1, seed=42):
    assert train_ratio + val_ratio < 1
    sentence_copy = sentences.copy()
    random.Random(seed).shuffle(sentence_copy)
    n = len(sentence_copy)
    train_end = int(train_ratio * n)
    val_end = int(train_ratio * n + val_ratio * n)
    train_sentences = sentence_copy[:train_end]
    val_sentences = sentence_copy[train_end:val_end]
    test_sentences = sentence_copy[val_end:]
    return train_sentences, val_sentences, test_sentences


In [48]:
def save_vocab(word2idx, idx2word, path):
    dirname = os.path.dirname(path)
    if dirname:
        os.makedirs(dirname, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"word2idx": word2idx, "idx2word": idx2word}, f, ensure_ascii=False, indent=2)
    print(f"vocab saved to {path}")


def load_vocab(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    word2idx = data["word2idx"]
    idx2word_temp = data["idx2word"]
    idx2word = {}
    for k,v in idx2word_temp.items():
        idx2word[int(k)] = v
    return word2idx, idx2word

def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, config, vocab_path, path):
    dirname = os.path.dirname(path)
    if dirname:
        os.makedirs(dirname, exist_ok=True)
    checkpoint = {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "config": config,
            "vocab_path": vocab_path,
        }
    torch.save(checkpoint, path)
    print(f"checkpoint saved to {path}")

def load_checkpoint(model, optimizer, path, map_location="cpu"):
    checkpoint = torch.load(path, map_location=map_location)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return checkpoint

In [ ]:
# === Real corpus 1-epoch smoke training ===
config = {
    "seed": 42,
    "data_dir": "/Users/dook/Desktop/保研项目/semantic-communication-cuc/code/data/raw/europarl-v7/txt/en",
    "max_sentences":20000,
    "min_len": 4,
    "max_len": 30,
    "min_freq": 2,
    "train_ratio": 0.8,
    "val_ratio": 0.1,
    "embed_dim": 128,
    "hidden_dim": 256,
    "num_layers": 1,
    "batch_size": 64,
    "lr": 1e-3,
    "epochs": 20,
    "vocab_path": "experiments/lstm_noiseless_20k/vocab.json",
    "checkpoint_path": "experiments/lstm_noiseless_20k/checkpoint_epoch20.pt",
    "best_checkpoint_path": "experiments/lstm_noiseless_20k/checkpoint_best.pt",
}

device = get_device()
config["device"] = str(device)
set_seed(config["seed"])
real_sentences = load_corpus_from_dir(
    config["data_dir"],
    min_len=config["min_len"],
    max_len=config["max_len"],
    max_sentences=config["max_sentences"],
)

train_sentences, val_sentences, test_sentences = split_sentences(
    real_sentences,
    train_ratio=config["train_ratio"],
    val_ratio=config["val_ratio"],
    seed=config["seed"],
)

word2idx, idx2word = build_vocab(train_sentences, min_freq=config["min_freq"])
save_vocab(word2idx, idx2word, config["vocab_path"])

train_dataset = TextDataset(train_sentences, word2idx)
val_dataset = TextDataset(val_sentences, word2idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    collate_fn=collate_fn,
)

vocab_size = len(word2idx)
pad_idx = word2idx["<PAD>"]

config["vocab_size"] = vocab_size
config["pad_idx"] = pad_idx

encoder = Encoder(
    vocab_size=vocab_size,
    embed_dim=config["embed_dim"],
    hidden_dim=config["hidden_dim"],
    num_layers=config["num_layers"],
    pad_idx=pad_idx,
)

decoder = Decoder(
    vocab_size=vocab_size,
    embed_dim=config["embed_dim"],
    hidden_dim=config["hidden_dim"],
    num_layers=config["num_layers"],
    pad_idx=pad_idx,
)

model = Seq2Seq(encoder, decoder).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

best_val_loss = float("inf")
for epoch in range(config["epochs"]):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_eval_loss = evaluate_loss(model, train_loader, criterion, device)
    val_loss = evaluate_loss(model, val_loader, criterion, device)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch + 1,
            train_loss=train_loss,
            val_loss=val_loss,
            config=config,
            vocab_path=config["vocab_path"],
            path=config["best_checkpoint_path"],
       )
    print(
        f"Epoch {epoch + 1}/{config['epochs']}, "
        f"Train Loss: {train_loss:.4f}, "
        f"Train Eval Loss: {train_eval_loss:.4f}, "
        f"Val Loss: {val_loss:.4f}"
    )

save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=config["epochs"],
    train_loss=train_loss,
    val_loss=val_loss,
    config=config,
    vocab_path=config["vocab_path"],
    path=config["checkpoint_path"],
)

vocab saved to experiments/lstm_noiseless_5k/vocab.json
checkpoint saved to experiments/lstm_noiseless_5k/checkpoint_best.pt
Epoch 1/100, Train Loss: 6.1559, Train Eval Loss: 5.3871, Val Loss: 5.0972
checkpoint saved to experiments/lstm_noiseless_5k/checkpoint_best.pt
Epoch 2/100, Train Loss: 5.1530, Train Eval Loss: 4.8369, Val Loss: 4.5638
checkpoint saved to experiments/lstm_noiseless_5k/checkpoint_best.pt
Epoch 3/100, Train Loss: 4.6747, Train Eval Loss: 4.4267, Val Loss: 4.1867
checkpoint saved to experiments/lstm_noiseless_5k/checkpoint_best.pt
Epoch 4/100, Train Loss: 4.3236, Train Eval Loss: 4.1071, Val Loss: 3.9126
checkpoint saved to experiments/lstm_noiseless_5k/checkpoint_best.pt
Epoch 5/100, Train Loss: 4.0335, Train Eval Loss: 3.8524, Val Loss: 3.7105


KeyboardInterrupt: 